In [1]:
from tensorflow.examples.tutorials.mnist import input_data
mnist = input_data.read_data_sets("MNIST_data/", one_hot=True)

Extracting MNIST_data/train-images-idx3-ubyte.gz
Extracting MNIST_data/train-labels-idx1-ubyte.gz
Extracting MNIST_data/t10k-images-idx3-ubyte.gz
Extracting MNIST_data/t10k-labels-idx1-ubyte.gz


In [2]:
import tensorflow as tf

# Interactive Session 을 사용
sess = tf.InteractiveSession()

In [3]:
# Create weight
def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev = 0.1)
    return tf.Variable(initial)

# Create bias
def bias_variable(shape):
    initial = tf.constant(0.1, shape = shape)
    return tf.Variable(initial)

# Convolution layer
def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides = [1, 1, 1, 1], padding = 'SAME')

# Pooling layer
def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize = [1, 2, 2, 1], strides = [1, 2, 2, 1], padding = 'SAME')

In [4]:
# Placeholders
# None: 배치 사이즈, 784: 28x28 이미지의 평탄 벡터
x = tf.placeholder(tf.float32, shape = [None, 784])
# None: 배치 사이즈, 10: 0~9까지를 표현하는 One-hot 벡터
y_ = tf.placeholder(tf.float32, shape = [None, 10])
# Dropout
keep_prob = tf.placeholder(tf.float32)

# 2D > 4D 로 shape 변경
x_image = tf.reshape(x, [-1, 28, 28, 1])

# Variables
W_conv1 = weight_variable([5, 5, 1, 32])
b_conv1 = bias_variable([32])
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])

# Convolution layer #1
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
# Pooling layer #1
h_pool1 = max_pool_2x2(h_conv1)

# Convolution layer #2
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
# Pooling layer #2
h_pool2 = max_pool_2x2(h_conv2)

# 4D > 2D 로 shape 변경
h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])

# Fully-connected layer #1
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)

# Dropout 을 적용
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# Fully-connected layer #2
y_conv = tf.matmul(h_fc1_drop, W_fc2) + b_fc2

In [5]:
# Loss function
cross_entropy = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(y_conv, y_))

# Training step
train_step = tf.train.AdamOptimizer(1e-4).minimize(cross_entropy)

# Initialize variables
sess.run(tf.global_variables_initializer())

# Training the model
for i in range(20000):
    batch = mnist.train.next_batch(50)
    train_step.run(feed_dict = {x: batch[0], y_: batch[1], keep_prob: 0.5})

In [6]:
# 예측값과 라벨값을 비교한 벡터
correct_prediction = tf.equal(tf.argmax(y_conv, 1), tf.argmax(y_, 1))

# 논리값 벡터를 실수로 변환하여 평균 = 정확도
accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

print(accuracy.eval(feed_dict = {x: mnist.test.images, y_: mnist.test.labels, keep_prob: 1.0}))

0.992
